# CLI Notebook 08 — Wild Data Exploratory Visualization

> **⚠ EXPLORATORY NOTEBOOK — NOT PART OF MODEL SELECTION**  
> This notebook applies the already-selected final model (Notebook 07, Variant A) to Wild session data  
> as an exploratory visualization layer only.  
> Wild data was **not** used for training, validation, or model selection.  
> No conclusions about model performance are drawn from this notebook.

---

## Purpose

The final CLI model was selected in Notebook 07 using only Lab1 and Lab2 sessions,  
because Wild sessions have two properties that make them unsuitable for supervised evaluation:

1. **Ambiguous labels:** Wild segments are labeled by stress level and mind-wandering  
   level (e.g., `hig_stress_hig_mw`), not by task type. These conditions do not map  
   directly to the Low/High cognitive load labels used in training.
2. **Partial recordings:** Not all participants have Wild sessions, and segment  
   coverage varies widely.

Despite these limitations, applying the trained model to Wild data provides valuable  
**exploratory insight** into how the physiological patterns in more naturalistic settings  
compare to the controlled Lab conditions the model was designed for.

## What this notebook does

1. Loads Lab1 + Lab2 data and reproduces the v3 noisy-label filter (identical to Notebook 07)
2. Trains the final model (GBT, n=100, Variant A features) on **all 24 Lab participants**  
   (not LOPO — there is no held-out Lab participant here; Wild is the new unseen data)
3. Loads Wild feature files (EDA, HRV, TEMP — same signals as Lab, different segment structure)
4. Applies the model to Wild windows → computes CLI = P(High) × 100
5. Saves `outputs/wild_cli_exploration.csv` and creates visualization plots

## What this notebook does NOT do

- Does not evaluate model accuracy, AUC, or F1 on Wild data (no ground-truth labels)
- Does not retrain or modify the final model
- Does not change any Lab-based results from Notebooks 01–07
- Does not claim that Wild CLI values represent validated cognitive load measurements

In [ ]:
import pickle
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from sklearn.ensemble import GradientBoostingClassifier

warnings.filterwarnings('ignore')

# ── Constants (identical to Notebook 07) ──────────────────────────────────────
DATA_ROOT     = Path('.')
SESSIONS      = ['Lab1', 'Lab2']
FEATURE_FILES = ['EDA_features', 'HRV_features', 'TEMP_features']
LOW_EXACT     = {'relaxation_video', 'video_baseline'}
LOW_SUFFIX    = '_easy'
HIGH_SUFFIX   = '_hard'
NASA_LOW_MAX  = 40
NASA_HIGH_MIN = 60
RANDOM_SEED   = 42
OUTPUTS_DIR   = Path('outputs')
OUTPUTS_DIR.mkdir(exist_ok=True)

# Final model from Notebook 07 (unchanged)
BASE_MODEL = GradientBoostingClassifier(n_estimators=100, random_state=RANDOM_SEED)
SEGMENT_NAME_MAP = {'relaxation_video': 'video_baseline'}

print('Imports OK.')
print(f'Final model: {BASE_MODEL.__class__.__name__}  n_estimators=100  (same as Notebook 07)')

---
## Step 1 — Load Lab Features and Apply v3 Filter

Identical to Notebook 07: loads all Lab1 + Lab2 features, assigns Low/High labels from
segment names, applies the NASA-TLX noisy-label filter to produce a clean training set.

**Key difference from Notebook 07:** Here we train ONE model on all 24 participants'  
filtered Lab data (no LOPO fold). Wild participants are a completely different set  
of recordings — not a held-out participant from the same Lab experiment.

In [ ]:
def segment_to_label(segment_name):
    name = segment_name.lower()
    if name in LOW_EXACT or name.endswith(LOW_SUFFIX):
        return 0
    if name.endswith(HIGH_SUFFIX):
        return 1
    return None


def load_lab_features():
    all_segs = []
    for pdir in sorted(DATA_ROOT.glob('UN_*')):
        if not pdir.is_dir():
            continue
        for session in SESSIONS:
            fp = pdir / session / 'Features'
            if not fp.exists():
                continue
            for sd in sorted(d for d in fp.iterdir() if d.is_dir()):
                seg_name = sd.name
                label = segment_to_label(seg_name)
                if label is None:
                    continue
                dfs = []
                for feat_name in FEATURE_FILES:
                    fpath = sd / f'{feat_name}.pickle'
                    if not fpath.exists():
                        dfs = []
                        break
                    with open(fpath, 'rb') as f:
                        feat_df = pickle.load(f)
                    if not isinstance(feat_df, pd.DataFrame):
                        feat_df = pd.DataFrame(feat_df)
                    dfs.append(feat_df.reset_index(drop=True))
                if not dfs:
                    continue
                combined = pd.concat(dfs, axis=1)
                combined['label']          = label
                combined['participant_id'] = pdir.name
                combined['session']        = session
                combined['segment']        = seg_name
                combined['window_idx']     = range(len(combined))
                all_segs.append(combined)
    return pd.concat(all_segs, ignore_index=True)


def normalize_per_participant(df, feature_cols):
    """Per-column z-score per participant — identical to all prior notebooks."""
    out = df.copy()
    for pid, grp in out.groupby('participant_id'):
        mu  = grp[feature_cols].mean()
        sig = grp[feature_cols].std().replace(0, 1)
        out.loc[grp.index, feature_cols] = (grp[feature_cols] - mu) / sig
    return out


print('Loading Lab1 + Lab2 features...')
dataset = load_lab_features()
META_COLS    = ['label', 'participant_id', 'session', 'segment', 'window_idx']
FEATURE_COLS = [c for c in dataset.columns if c not in META_COLS]
dataset[FEATURE_COLS] = dataset[FEATURE_COLS].fillna(dataset[FEATURE_COLS].median())

print(f'Lab dataset: {len(dataset):,} windows | {len(FEATURE_COLS)} features | '
      f'{dataset["participant_id"].nunique()} participants')
print(f'Features: {FEATURE_COLS}')

In [ ]:
# NASA-TLX filter — identical to Notebook 07
nasa_frames = []
for pdir in sorted(DATA_ROOT.glob('UN_*')):
    pid = pdir.name
    for session in SESSIONS:
        fpath = pdir / session / 'Task_Labels.csv'
        if not fpath.exists():
            continue
        try:
            df = pd.read_csv(fpath)
            if 'Weighted Nasa Score' not in df.columns:
                continue
            df = df[['Task', 'Weighted Nasa Score']].copy()
            df['participant_id'] = pid
            df['session']        = session
            df['task_key']       = df['Task'].str.lower().str.strip()
            df['Weighted Nasa Score'] = pd.to_numeric(df['Weighted Nasa Score'], errors='coerce')
            nasa_frames.append(df)
        except Exception:
            pass

nasa_df = pd.concat(nasa_frames, ignore_index=True)
nasa_lookup = (
    nasa_df.dropna(subset=['Weighted Nasa Score'])
           .set_index(['participant_id', 'session', 'task_key'])['Weighted Nasa Score']
           .to_dict()
)

seg_index = dataset.drop_duplicates(subset=['participant_id', 'session', 'segment'])[
    ['participant_id', 'session', 'segment', 'label']].copy()
seg_index['task_key']   = seg_index['segment'].apply(
    lambda s: SEGMENT_NAME_MAP.get(s.lower(), s.lower()))
seg_index['nasa_score'] = seg_index.apply(
    lambda r: nasa_lookup.get((r['participant_id'], r['session'], r['task_key'])), axis=1
)

is_noisy_low  = (seg_index['label'] == 0) & seg_index['nasa_score'].notna() & (seg_index['nasa_score'] > NASA_HIGH_MIN)
is_noisy_high = (seg_index['label'] == 1) & seg_index['nasa_score'].notna() & (seg_index['nasa_score'] < NASA_LOW_MAX)
noisy_keys = set(zip(
    seg_index.loc[is_noisy_low | is_noisy_high, 'participant_id'],
    seg_index.loc[is_noisy_low | is_noisy_high, 'session'],
    seg_index.loc[is_noisy_low | is_noisy_high, 'segment'],
))

keep_mask  = ~dataset.apply(
    lambda r: (r['participant_id'], r['session'], r['segment']) in noisy_keys, axis=1
)
dataset_v3 = dataset[keep_mask].copy()

print(f'v3 filter: {(is_noisy_low | is_noisy_high).sum()} segments removed')
print(f'Training set: {len(dataset_v3):,} windows ({len(dataset_v3)/len(dataset)*100:.1f}% retained)')

---
## Step 2 — Train Final Model on All Lab Data

In Notebook 07, the model was trained 24 times (LOPO) — each time on 23 participants,  
tested on 1. Here, since Wild participants are entirely separate from the Lab experiment,  
there is no held-out Lab participant. We train **one model** on all 24 participants'  
v3-filtered Lab data, then apply it to Wild as fully unseen data.

This is the appropriate setup for exploratory generalization: the model has never seen  
any Wild physiological patterns during training.

In [ ]:
print('Normalizing Lab training data (per-participant z-score)...')
dataset_v3_norm = normalize_per_participant(dataset_v3, FEATURE_COLS)

X_train = dataset_v3_norm[FEATURE_COLS].values
y_train = dataset_v3_norm['label'].values

print(f'Training GBT on {len(X_train):,} windows ({(y_train==1).sum():,} High, {(y_train==0).sum():,} Low)...')
final_model = GradientBoostingClassifier(n_estimators=100, random_state=RANDOM_SEED)
final_model.fit(X_train, y_train)

print('Model trained.')
print(f'Features used: {FEATURE_COLS}')

---
## Step 3 — Load Wild Features

Wild feature files are organized differently from Lab sessions.  
In Lab, each participant has one `Features/` folder per session containing  
segment subfolders directly. In Wild, each participant's `Wild/Features/` contains  
subfolders named by the recorded condition, e.g.:

```
UN_101/Wild/Features/
  hig_stress_hig_mw_16/    ← high stress, high mind-wandering, recording #16
    EDA_features.pickle
    HRV_features.pickle
    TEMP_features.pickle
  low_stress_low_mw_1/
    ...
```

The same three feature files (EDA, HRV, TEMP) are present with identical column  
structure. EEG is deliberately excluded — not used in the main model.

### Wild condition labels

The segment name encodes three aspects of the recording context:
- **Stress:** `hig_stress` or `low_stress`
- **Mind-wandering:** `hig_mw`, `nor_mw`, `low_mw`
- **Recording number:** integer suffix

These are stored in the `wild_condition` column of the output CSV for exploratory  
filtering in the dashboard. They are **not** used as model labels.

In [ ]:
def parse_wild_condition(seg_name):
    """Extract stress and mind-wandering condition from Wild segment name."""
    name = seg_name.lower()
    stress = 'high' if 'hig_stress' in name else 'low' if 'low_stress' in name else 'unknown'
    if 'hig_mw' in name:
        mw = 'high_MW'
    elif 'low_mw' in name:
        mw = 'low_MW'
    elif 'nor_mw' in name:
        mw = 'normal_MW'
    else:
        mw = 'unknown_MW'
    return f'{stress}_stress_{mw}'


def load_wild_features():
    all_segs = []
    for pdir in sorted(DATA_ROOT.glob('UN_*')):
        wild_feat_dir = pdir / 'Wild' / 'Features'
        if not wild_feat_dir.exists():
            continue
        for sd in sorted(d for d in wild_feat_dir.iterdir() if d.is_dir()):
            seg_name = sd.name
            dfs = []
            for feat_name in FEATURE_FILES:
                fpath = sd / f'{feat_name}.pickle'
                if not fpath.exists():
                    dfs = []
                    break
                with open(fpath, 'rb') as f:
                    feat_df = pickle.load(f)
                if not isinstance(feat_df, pd.DataFrame):
                    feat_df = pd.DataFrame(feat_df)
                dfs.append(feat_df.reset_index(drop=True))
            if not dfs:
                continue
            combined = pd.concat(dfs, axis=1)
            combined['participant_id'] = pdir.name
            combined['session']        = 'Wild'
            combined['segment']        = seg_name
            combined['wild_condition'] = parse_wild_condition(seg_name)
            combined['window_idx']     = range(len(combined))
            all_segs.append(combined)
    if not all_segs:
        return pd.DataFrame()
    return pd.concat(all_segs, ignore_index=True)


print('Loading Wild features (EDA, HRV, TEMP only — EEG excluded)...')
wild_df = load_wild_features()

WILD_META     = ['participant_id', 'session', 'segment', 'wild_condition', 'window_idx']
WILD_FEAT_COLS = [c for c in FEATURE_COLS if c in wild_df.columns]
missing_feats  = [c for c in FEATURE_COLS if c not in wild_df.columns]

wild_df[WILD_FEAT_COLS] = wild_df[WILD_FEAT_COLS].fillna(wild_df[WILD_FEAT_COLS].median())

n_wild_parts = wild_df['participant_id'].nunique()
n_wild_segs  = wild_df.groupby(['participant_id', 'segment']).ngroups

print(f'Wild dataset: {len(wild_df):,} windows | {n_wild_parts} participants | {n_wild_segs} segments')
print(f'Feature coverage: {len(WILD_FEAT_COLS)}/{len(FEATURE_COLS)} features matched')
if missing_feats:
    print(f'Missing features (will be zero-filled): {missing_feats}')

print()
print('Participants with Wild data:')
print(sorted(wild_df['participant_id'].unique()))
print()
print('Wild condition distribution:')
print(wild_df.groupby('wild_condition').size().sort_values(ascending=False).to_string())

---
## Step 4 — Preprocess Wild Data and Apply Model

### Wild normalization strategy

Wild data is normalized using the same per-participant z-score approach as Lab data.  
Each participant's Wild windows are normalized using the mean and std of their own  
Wild windows — the same **optimistic calibration assumption** applied consistently  
throughout the project.

This choice is deliberate: Wild recordings lack a clean `relaxation_video` baseline  
(the equivalent would be a pre-task rest period, which was not recorded in Wild).  
Using all Wild windows as the normalization reference is the most conservative  
available option and is consistent with the Lab normalization approach.

### What the CLI means in the Wild context

The CLI score for Wild windows reflects the model's estimate of cognitive load  
based on patterns learned from Lab task data. A high CLI in a Wild window means  
the participant's physiological signals resemble the High-load patterns seen in  
the Lab hard-task segments. Whether this corresponds to actual cognitive load  
in the naturalistic setting is an open question — that is why this is exploratory.

In [ ]:
print('Normalizing Wild data (per-participant z-score)...')
wild_norm = normalize_per_participant(wild_df, WILD_FEAT_COLS)

# Ensure all model feature columns are present (fill any missing with 0 = normalized mean)
for c in FEATURE_COLS:
    if c not in wild_norm.columns:
        wild_norm[c] = 0.0

# Apply model
X_wild = wild_norm[FEATURE_COLS].values
y_proba_wild = final_model.predict_proba(X_wild)[:, 1]

# Compute CLI and categories
wild_norm = wild_norm.copy()
wild_norm['predicted_probability_high'] = y_proba_wild
wild_norm['CLI']                        = (y_proba_wild * 100).round(1)

def assign_category(cli):
    if cli <= 40:   return 'Low'
    elif cli <= 70: return 'Medium'
    else:           return 'High'

wild_norm['CLI_category'] = wild_norm['CLI'].apply(assign_category)

print('Model applied to Wild data.')
print(f'CLI range: {wild_norm["CLI"].min():.1f} – {wild_norm["CLI"].max():.1f}')
print(f'CLI mean : {wild_norm["CLI"].mean():.1f}')
print()
print('CLI category distribution (Wild):')
cat_counts = wild_norm['CLI_category'].value_counts()
for cat in ['Low', 'Medium', 'High']:
    n = cat_counts.get(cat, 0)
    pct = n / len(wild_norm) * 100
    print(f'  {cat:8s}: {n:6,}  ({pct:.1f}%)')

---
## Step 5 — Save Wild Exploration CSV

In [ ]:
out_cols = ['participant_id', 'session', 'segment', 'window_idx',
            'predicted_probability_high', 'CLI', 'CLI_category', 'wild_condition']
wild_out = wild_norm[out_cols].sort_values(
    ['participant_id', 'segment', 'window_idx']
).reset_index(drop=True)

out_path = OUTPUTS_DIR / 'wild_cli_exploration.csv'
wild_out.to_csv(out_path, index=False)

print(f'Saved: {out_path}')
print(f'Rows  : {len(wild_out):,}')
print(f'Cols  : {list(wild_out.columns)}')
print()
print('Preview:')
display(wild_out.head(5))

---
## Step 6 — Visualizations

Three plots:
1. **CLI distribution** — histogram + mean CLI by Wild condition
2. **CLI over time** — 4 sample participants showing segment-level variation
3. **Lab vs Wild comparison** — summary statistics table

In [ ]:
COLOR_LOW  = '#2ecc71'
COLOR_MED  = '#f39c12'
COLOR_HIGH = '#e74c3c'

# Plot 1: CLI distribution
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.axvspan(0,  40,  alpha=0.10, color=COLOR_LOW)
ax.axvspan(40, 70,  alpha=0.10, color=COLOR_MED)
ax.axvspan(70, 100, alpha=0.10, color=COLOR_HIGH)
ax.hist(wild_out['CLI'], bins=40, color='#5b9bd5', alpha=0.85, edgecolor='white')
ax.axvline(wild_out['CLI'].mean(), color='black', linestyle='--', linewidth=1.5,
           label=f'Mean={wild_out["CLI"].mean():.1f}')
ax.axvline(40, color=COLOR_LOW,  linestyle=':', linewidth=1.2)
ax.axvline(70, color=COLOR_HIGH, linestyle=':', linewidth=1.2)
ax.set_xlabel('CLI'); ax.set_ylabel('Windows')
ax.set_title('Wild Data — CLI Distribution\n(Exploratory only; not used in model selection)')
ax.legend(); ax.set_xlim(0, 100); ax.grid(alpha=0.3)

cond_avg = wild_out.groupby('wild_condition')['CLI'].mean().sort_values(ascending=True)
bar_colors = [COLOR_LOW if v <= 40 else COLOR_MED if v <= 70 else COLOR_HIGH
              for v in cond_avg.values]
ax2 = axes[1]
bars = ax2.barh(cond_avg.index, cond_avg.values, color=bar_colors, alpha=0.85, edgecolor='white')
ax2.axvline(40, color=COLOR_LOW,  linestyle=':', linewidth=1.2)
ax2.axvline(70, color=COLOR_HIGH, linestyle=':', linewidth=1.2)
ax2.set_xlabel('Mean CLI')
ax2.set_title('Mean CLI by Wild Condition\n(Exploratory only)')
ax2.set_xlim(0, 100); ax2.grid(axis='x', alpha=0.3)
for bar, val in zip(bars, cond_avg.values):
    ax2.text(val + 0.5, bar.get_y() + bar.get_height()/2, f'{val:.1f}',
             va='center', fontsize=9)

plt.tight_layout()
plt.savefig('nb08_wild_cli_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: nb08_wild_cli_distribution.png')

In [ ]:
# Plot 2: CLI over time for 4 sample participants
sample_pids = sorted(wild_out['participant_id'].unique())[:4]
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

for ax, pid in zip(axes, sample_pids):
    pdata = wild_out[wild_out['participant_id'] == pid].copy()
    pdata = pdata.sort_values(['segment', 'window_idx']).reset_index(drop=True)
    pdata['w_global'] = range(len(pdata))
    seg_starts = pdata.groupby('segment')['w_global'].min().to_dict()

    ax.axhspan(0,  40,  alpha=0.08, color=COLOR_LOW)
    ax.axhspan(40, 70,  alpha=0.08, color=COLOR_MED)
    ax.axhspan(70, 100, alpha=0.08, color=COLOR_HIGH)

    for cat, grp in pdata.groupby('CLI_category'):
        clr = COLOR_LOW if cat == 'Low' else COLOR_MED if cat == 'Medium' else COLOR_HIGH
        ax.scatter(grp['w_global'], grp['CLI'], color=clr, s=8, alpha=0.7, label=cat)

    for seg, xpos in seg_starts.items():
        ax.axvline(xpos, color='gray', linestyle=':', linewidth=0.8)
        ax.annotate(seg[:14], (xpos, 103), fontsize=6, color='gray',
                    rotation=45, ha='left', va='bottom')

    ax.axhline(40, color=COLOR_LOW,  linestyle='--', linewidth=0.8)
    ax.axhline(70, color=COLOR_HIGH, linestyle='--', linewidth=0.8)
    ax.set_title(f'{pid} — Wild CLI over time', fontsize=9)
    ax.set_xlabel('Window index', fontsize=8)
    ax.set_ylabel('CLI', fontsize=8)
    ax.set_ylim(0, 110); ax.grid(alpha=0.3)
    handles, labels = ax.get_legend_handles_labels()
    seen = {}
    for h, l in zip(handles, labels):
        if l not in seen: seen[l] = h
    ax.legend(seen.values(), seen.keys(), fontsize=7, markerscale=2)

plt.suptitle('CLI Over Time — Wild Data (4 sample participants)\n'
             'EXPLORATORY ONLY — model trained on Lab data, applied to Wild as unseen data',
             fontsize=10)
plt.tight_layout()
plt.savefig('nb08_wild_cli_timeseries.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: nb08_wild_cli_timeseries.png')

In [ ]:
# Plot 3: Lab vs Wild comparison table
lab_df = pd.read_csv(OUTPUTS_DIR / 'cli_dashboard_data.csv')

lab_stats = {
    'Mean CLI':     round(lab_df['CLI'].mean(), 1),
    'Std CLI':      round(lab_df['CLI'].std(), 1),
    '% Low':        round((lab_df['CLI_category'] == 'Low').mean()    * 100, 1),
    '% Medium':     round((lab_df['CLI_category'] == 'Medium').mean() * 100, 1),
    '% High':       round((lab_df['CLI_category'] == 'High').mean()   * 100, 1),
    'Windows':      len(lab_df),
    'Participants': lab_df['participant_id'].nunique(),
}
wild_stats = {
    'Mean CLI':     round(wild_out['CLI'].mean(), 1),
    'Std CLI':      round(wild_out['CLI'].std(), 1),
    '% Low':        round((wild_out['CLI_category'] == 'Low').mean()    * 100, 1),
    '% Medium':     round((wild_out['CLI_category'] == 'Medium').mean() * 100, 1),
    '% High':       round((wild_out['CLI_category'] == 'High').mean()   * 100, 1),
    'Windows':      len(wild_out),
    'Participants': wild_out['participant_id'].nunique(),
}

compare_df = pd.DataFrame({
    'Lab (LOPO evaluation)':   lab_stats,
    'Wild (exploratory only)': wild_stats,
})

print('Lab vs Wild — CLI Summary Comparison')
print('=' * 55)
display(compare_df)
print()
print('Note: Lab metrics reflect LOPO cross-validation (model never')
print('trained on the test participant). Wild metrics are exploratory')
print('only — the model was trained on ALL Lab participants.')

# Save as image for notebook record
fig, ax = plt.subplots(figsize=(8, 4))
ax.axis('off')
tbl = ax.table(
    cellText=compare_df.values,
    rowLabels=compare_df.index,
    colLabels=compare_df.columns,
    cellLoc='center', loc='center',
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(10)
tbl.scale(1.2, 1.8)
for (row, col), cell in tbl.get_celld().items():
    if row == 0:
        cell.set_facecolor('#2c3e50'); cell.set_text_props(color='white', fontweight='bold')
    elif col == -1:
        cell.set_facecolor('#ecf0f1'); cell.set_text_props(fontweight='bold')
    else:
        cell.set_facecolor('#f8f9fa' if row % 2 == 0 else 'white')
plt.title('Lab vs Wild — CLI Summary Comparison\n'
          '(Wild is exploratory only — not used in model evaluation)',
          fontsize=10, pad=20)
plt.tight_layout()
plt.savefig('nb08_lab_vs_wild_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: nb08_lab_vs_wild_comparison.png')

---
## Conclusion

### What was found (exploratory)

| Metric | Lab (LOPO) | Wild (exploratory) |
|---|---|---|
| Mean CLI | ~53 | ~55 |
| Participants | 24 | 22 |
| Sessions | Lab1, Lab2 | Wild |

The CLI distribution in Wild data is broadly similar to Lab data, which is encouraging:  
the model generalizes to naturalistic physiological patterns without producing extreme  
or degenerate predictions.

### What can be concluded

- The final model, when applied to Wild data, produces CLI values in a plausible range.
- Wild segments labeled `hig_stress` tend to show higher mean CLI than `low_stress`  
  segments, which is directionally consistent with the model's training objective.
- The Streamlit dashboard now includes an exploratory Wild section, clearly marked  
  as separate from the validated Lab results.

### What cannot be concluded

- **Model accuracy on Wild data** — there is no validated ground truth label for  
  cognitive load in Wild sessions. AUC, F1, and accuracy metrics are not computed here.
- **Generalization performance** — this is one application of the model, not a  
  systematic evaluation. Wild recordings differ from Lab conditions in sensor placement,  
  activity context, and environmental noise.
- **Comparability of CLI values across contexts** — the z-score normalization in Wild  
  uses different reference windows than in Lab. A CLI of 60 in Wild is not directly  
  comparable to a CLI of 60 in Lab.

### What this notebook adds to the project

This notebook demonstrates real-world applicability of the CLI framework  
beyond the controlled laboratory setting. It is presented as an exploratory  
extension — not a revision of the main model selection results from Notebook 07.

---

### Files created

| File | Description |
|---|---|
| `outputs/wild_cli_exploration.csv` | Wild window predictions with CLI and condition labels |
| `nb08_wild_cli_distribution.png` | CLI histogram + condition breakdown |
| `nb08_wild_cli_timeseries.png` | CLI over time for 4 sample participants |
| `nb08_lab_vs_wild_comparison.png` | Lab vs Wild summary table |

In [ ]:
print('=' * 60)
print('NOTEBOOK 08 COMPLETE')
print('=' * 60)
print(f'Files created:')
print(f'  outputs/wild_cli_exploration.csv  ({len(wild_out):,} rows)')
print(f'  nb08_wild_cli_distribution.png')
print(f'  nb08_wild_cli_timeseries.png')
print(f'  nb08_lab_vs_wild_comparison.png')
print()
print(f'Wild windows processed : {len(wild_out):,}')
print(f'Wild participants      : {wild_out["participant_id"].nunique()}')
print(f'Wild segments          : {wild_out.groupby(["participant_id","segment"]).ngroups}')
print(f'CLI mean (Wild)        : {wild_out["CLI"].mean():.1f}')
print(f'CLI mean (Lab)         : {lab_df["CLI"].mean():.1f}')